# CSI4142 Assignment 1

1. Andrew Pham - 300226985
2. Kevin Yao - 300295024

# Part 1 - Validity Checker

### Dataset

* Name: NYC Property Sales
* Author: City of New York
* Purpose: A record of every building or unit sold in NYC over a period of 1 year.
* Shape: 84548 rows, 22 columns

https://www.kaggle.com/datasets/new-york-city/nyc-property-sales

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

/opt/miniconda3/envs/csi4142/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download latest version
file_path = "nyc-rolling-sales.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "new-york-city/nyc-property-sales", path=file_path
)
orig_df = df.copy()

In [3]:
df.head()

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,...,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,...,5,0,5,1633,6440,1900,2,C2,6625000,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,,C7,234 EAST 4TH STREET,...,28,3,31,4616,18690,1900,2,C7,-,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,,C7,197 EAST 3RD STREET,...,16,1,17,2212,7803,1900,2,C7,-,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,...,10,0,10,2272,6794,1913,2,C4,3936272,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,...,6,0,6,2369,4615,1900,2,C2,8000000,2016-11-17 00:00:00


In [4]:
df.describe()

,Unnamed: 0,BOROUGH,BLOCK,LOT,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,YEAR BUILT,TAX CLASS AT TIME OF SALE
count,84548.000000,84548.000000,84548.000000,84548.000000,84548.000000,84548.000000,84548.000000,84548.000000,84548.000000,84548.000000
mean,10344.359878,2.998758,4237.218976,376.224015,10731.991614,2.025264,0.193559,2.249184,1789.322976,1.657485
std,7151.779436,1.289790,3568.263407,658.136814,1290.879147,16.721037,8.713183,18.972584,537.344993,0.819341
min,4.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,4231.000000,2.000000,1322.750000,22.000000,10305.000000,0.000000,0.000000,1.000000,1920.000000,1.000000
50%,8942.000000,3.000000,3311.000000,50.000000,11209.000000,1.000000,0.000000,1.000000,1940.000000,2.000000
75%,15987.250000,4.000000,6281.000000,1001.000000,11357.000000,2.000000,0.000000,2.000000,1965.000000,2.000000
max,26739.000000,5.000000,16322.000000,9106.000000,11694.000000,1844.000000,2261.000000,2261.000000,2017.000000,4.000000


### Validity Check 1: Data Type Errors

**Type of error:** Data type inconsistency

**Description:** The `BOROUGH` column should be numeric (int), but some values are stored as strings. This breaks numeric validation and downstream analysis.

The error was introduced by randomly selecting 5% of rows in `BOROUGH` and converting their numeric values to strings (e.g., `3` → `'3'`).

In the real world, this type of error could be introduced through human error, or through errors in data collection pipelines such as migrations from previous string names to number codes.

In [5]:
# Introduce data type errors in BOROUGH (~5% as strings with quotes)
np.random.seed(1)
err_df = df.copy()
err_idx = err_df.sample(frac=0.05, random_state=1).index

# change type to object so we can mix ints and strings
err_df.BOROUGH = err_df.BOROUGH.astype(str)

# Convert sampled rows to strings with single quotes (e.g., 1 -> '1')
err_df.loc[err_idx, "BOROUGH"] = (
    err_df.loc[err_idx, "BOROUGH"].astype(str).map(lambda v: f"'{v}'")
)

# Show a few corrupted values
err_df.loc[err_idx, ["BOROUGH"]].head()

,BOROUGH
32184,'3'
77701,'5'
83718,'5'
56085,'4'
12994,'1'


In [6]:
def fix_borough_dtype(input_df):
    """
    Detect and correct data type errors in the 'BOROUGH' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'BOROUGH' column.

    Returns:
        output_df (pd.DataFrame): The corrected DataFrame.
        non_numeric_mask (pd.Series): A boolean mask of rows that were corrected.
    """
    output_df = input_df.copy()

    # Normalize by stripping single quotes from strings
    cleaned = output_df["BOROUGH"].astype(str).str.strip("'")

    # Detect non-numeric entries (strings, mixed types)
    non_numeric_mask = pd.to_numeric(cleaned, errors="coerce").isna()

    # Correct by coercing to numeric and restoring to int
    output_df["BOROUGH"] = pd.to_numeric(cleaned, errors="coerce").astype(int)
    return output_df, non_numeric_mask


# Run detection + correction
fixed_df, bad_mask = fix_borough_dtype(err_df)

In [7]:
# show 3 examples of bad values
print("Bad values before correction:")
display(err_df.loc[err_idx, ["BOROUGH"]].head(3))
print(f"BOROUGH dtype before correction: {err_df['BOROUGH'].dtype}")
# show 3 examples of corrected values
print("Corrected values after correction:")
display(fixed_df.loc[err_idx, ["BOROUGH"]].head(3))
print(f"BOROUGH dtype after correction: {fixed_df['BOROUGH'].dtype}")

Bad values before correction:


,BOROUGH
32184,'3'
77701,'5'
83718,'5'


BOROUGH dtype before correction: str
Corrected values after correction:


,BOROUGH
32184,3
77701,5
83718,5


BOROUGH dtype after correction: int64


**Results:** The function will correctly coerce string representations of numbers into integers, and the whole column will be set to the correct int64 dtype.

### Validity Check 2: Range Errors

**Type of error:** Out-of-range values

**Description:** The `YEAR BUILT` column should be between 1652 ([oldest NYC house](https://www.nypap.org/preservation-history/wyckoff-house/)) and 2026 (current year). Values outside this range are invalid. In fact, there are 6970 rows already in the dataset that have a year built of 0. Presumably, this is because it was unknown or not collected.

In [8]:
# count the number of rows that are out of range
df["YEAR BUILT"].value_counts()

YEAR BUILT
0       6970
1920    6045
1930    5043
1925    4312
1910    3585
        ... 
1829       1
1832       1
1849       1
1855       1
1680       1
Name: count, Length: 158, dtype: int64

**How the error was introduced:** Randomly selected ~5% of rows in `YEAR BUILT` and replaced them with out-of-range years (some too old, some in the future), randomly selected within 100 years before or after the range limits.

In [9]:
# Introduce range errors in YEAR BUILT (~5% out of range)
np.random.seed(2)
range_err_df = df.copy()
range_err_idx = range_err_df.sample(frac=0.05, random_state=2).index

# Split into too-old and future years
half = len(range_err_idx) // 2
old_idx = range_err_idx[:half]
future_idx = range_err_idx[half:]

old_age_possibilities = range(1552, 1652)
future_age_possibilities = range(2026, 2126)

# sample for each row

old_samples = np.random.choice(old_age_possibilities, size=len(old_idx))
future_samples = np.random.choice(future_age_possibilities, size=len(future_idx))

range_err_df.loc[old_idx, "YEAR BUILT"] = old_samples
range_err_df.loc[future_idx, "YEAR BUILT"] = future_samples

display(range_err_df.loc[old_idx][["YEAR BUILT"]].head(3))
display(range_err_df.loc[future_idx][["YEAR BUILT"]].head(3))

,YEAR BUILT
10385,1592
75998,1567
39590,1624


,YEAR BUILT
78933,2038
74903,2090
50984,2047


In [10]:
def find_erroneous_year_built_range(input_df, min_year=1652, max_year=2026):
    """
    Detect and return indices of out-of-range years in the 'YEAR BUILT' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'YEAR BUILT' column.
        min_year (int): The minimum valid year (default: 1652).
        max_year (int): The maximum valid year (default: 2026).

    Returns:
        out_of_range (pd.Series): A boolean mask of rows that were corrected.
    """
    output_df = input_df.copy()
    year_series = pd.to_numeric(output_df["YEAR BUILT"], errors="coerce")
    out_of_range = (year_series < min_year) | (year_series > max_year)

    return out_of_range


# Run detection + correction
out_of_range_mask = find_erroneous_year_built_range(range_err_df)
fixed_range_df = range_err_df[~out_of_range_mask]


print(f"Rows out of range detected: {out_of_range_mask.sum()}")
print("Bad values before correction:")
display(range_err_df.loc[out_of_range_mask, ["YEAR BUILT"]].head(3))
print("\nValue counts after filtering:")
fixed_range_df[["YEAR BUILT"]].describe()

Rows out of range detected: 10800
Bad values before correction:


,YEAR BUILT
9,2087
26,2051
49,1601



Value counts after filtering:


,YEAR BUILT
count,73748.000000
mean,1950.116749
std,34.218845
min,1680.000000
25%,1925.000000
50%,1945.000000
75%,1970.000000
max,2026.000000


**Qualitative results:** The detector flags any `YEAR BUILT` values outside 1652–2026 and drops those rows because the true year is unknown. In the sample output, invalid years (too old or too new) are identified and removed; the value counts after filtering show only valid years remain.

### Validity Check 3: Format Errors

**Type of error:** Invalid string format

**Description:** The `BUILDING CLASS AT TIME OF SALE` field should be exactly 2 characters: the first must be a capital letter, and the second must be either a single digit or another capital letter (pattern `^[A-Z][0-9A-Z]$`).

**How the error was introduced:** Randomly selected ~5% of rows and replaced `BUILDING CLASS AT TIME OF SALE` with invalid formats (e.g., lowercase, too long, or wrong character order).

In [11]:
# Introduce format errors in BUILDING CLASS AT TIME OF SALE (~5%)
np.random.seed(3)
format_err_df = df.copy()
format_err_idx = format_err_df.sample(frac=0.05, random_state=3).index

digits = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
letters = [chr(i) for i in range(65, 91)]
letters_lower = [chr(i) for i in range(97, 123)]


def create_invalid_format_lowercase_letter_2_char():
    chr1 = np.random.choice(letters_lower)
    chr2 = np.random.choice(letters)
    return chr1 + chr2


def create_invalid_format_lowercase_letter_1_char_1_digit():
    chr1 = np.random.choice(letters_lower)
    chr2 = np.random.choice(digits)
    return chr1 + chr2


def create_invalid_format_lowercase_letter_1_digit_1_char():
    chr1 = np.random.choice(digits)
    chr2 = np.random.choice(letters)
    return chr1 + chr2


format_err_fns = [
    create_invalid_format_lowercase_letter_2_char,
    create_invalid_format_lowercase_letter_1_char_1_digit,
    create_invalid_format_lowercase_letter_1_digit_1_char,
]

# randomly select one of the functions for each format_err_idx row and generate the invalid format
for err_idx in format_err_idx:
    invalid_format_fn = np.random.choice(format_err_fns)
    invalid_format = invalid_format_fn()
    format_err_df.loc[err_idx, "BUILDING CLASS AT TIME OF SALE"] = invalid_format

print("Example of bad values:")
format_err_df.loc[format_err_idx, ["BUILDING CLASS AT TIME OF SALE"]].head(3)

Example of bad values:


,BUILDING CLASS AT TIME OF SALE
24209,8Z
71811,iA
54599,t9


In [12]:
def find_erroneous_building_class_format(input_df):
    """
    Detect and return indices of invalid building class formats in the 'BUILDING CLASS AT TIME OF SALE' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'BUILDING CLASS AT TIME OF SALE' column.

    Returns:
        out_of_range (pd.Series): A boolean mask of rows that were corrected.
    """
    output_df = input_df.copy()
    cleaned = (
        output_df["BUILDING CLASS AT TIME OF SALE"].astype(str).str.strip().str.upper()
    )

    valid_mask = cleaned.str.match(r"^[A-Z][0-9A-Z]$")

    # Correct by normalizing valid values and setting invalids to NaN
    output_df["BUILDING CLASS AT TIME OF SALE"] = cleaned.where(valid_mask, np.nan)
    return ~valid_mask


# Run detection + correction
format_mask = find_erroneous_building_class_format(format_err_df)
fixed_format_df = format_err_df[~format_mask]

print(f"Rows with invalid format detected: {format_mask.sum()}")
print("Bad values before correction:")
display(format_err_df.loc[format_mask, ["BUILDING CLASS AT TIME OF SALE"]].head(3))
print("After correction:")
display(fixed_format_df.loc[format_mask, ["BUILDING CLASS AT TIME OF SALE"]].head(3))

Rows with invalid format detected: 1406
Bad values before correction:


,BUILDING CLASS AT TIME OF SALE
3,5C
85,3I
217,0J


After correction:


,BUILDING CLASS AT TIME OF SALE


**Qualitative results:** The detector flags any value not matching the pattern `^[A-Z][0-9A-Z]$` (NOT a uppercase letter followed by a digit or other upper case letter). The sample output shows invalid entries (e.g., lowercase or wrong length) and that they are dropped if invalid.

### Validity Check 4: Consistency Errors

**Type of error:** Inconsistent totals

**Description:** For each row, `RESIDENTIAL UNITS + COMMERCIAL UNITS` should equal `TOTAL UNITS`. Any mismatch indicates an inconsistency in the record.

**How the error was introduced:** Randomly selected ~5% of rows and altered `TOTAL UNITS` so it no longer equals `RESIDENTIAL UNITS + COMMERCIAL UNITS`.

In [13]:
# Introduce consistency errors in unit totals (~5%)
np.random.seed(4)
consistency_df = df.copy()
consistency_idx = consistency_df.sample(frac=0.05, random_state=4).index

# Break the consistency by subtracting 1 from TOTAL UNITS
consistency_df.loc[consistency_idx, "TOTAL UNITS"] = (
    consistency_df.loc[consistency_idx, "TOTAL UNITS"] - 1
)

# show 3 examples of bad values
print("Bad values before correction:")
display(
    consistency_df.loc[
        consistency_idx, ["RESIDENTIAL UNITS", "COMMERCIAL UNITS", "TOTAL UNITS"]
    ].head(3)
)

Bad values before correction:


,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS
51352,2,0,1
36593,2,0,1
80982,2,0,1


In [14]:
# Function to detect and correct consistency errors
def find_erroneous_unit_consistency(input_df):
    """
    Detect and return indices of inconsistent unit totals in the 'TOTAL UNITS' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'TOTAL UNITS' column.

    Returns:
        inconsistency_mask (pd.Series): A boolean mask of rows that were not consistent.
    """
    output_df = input_df.copy()
    expected_total = output_df["RESIDENTIAL UNITS"] + output_df["COMMERCIAL UNITS"]
    inconsistency_mask = output_df["TOTAL UNITS"] != expected_total

    return inconsistency_mask


# Run detection + correction
inconsistency_mask = find_erroneous_unit_consistency(consistency_df)
fixed_consistency_df = consistency_df[~inconsistency_mask]

print(f"Rows with inconsistent totals detected: {inconsistency_mask.sum()}")
print("Bad values before correction:")
display(
    consistency_df.loc[
        inconsistency_mask,
        ["RESIDENTIAL UNITS", "COMMERCIAL UNITS", "TOTAL UNITS"],
    ].head(3)
)
print("After correction:")
display(
    fixed_consistency_df.loc[
        inconsistency_mask,
        ["RESIDENTIAL UNITS", "COMMERCIAL UNITS", "TOTAL UNITS"],
    ].head(3)
)

Rows with inconsistent totals detected: 6636
Bad values before correction:


,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS
12,0,0,-1
18,0,0,-1
86,1,0,0


After correction:


,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS


**Qualitative results:** The detector flags rows where the computed total (`RESIDENTIAL UNITS + COMMERCIAL UNITS`) does not match `TOTAL UNITS`. The examples show mismatched totals before correction, then we drop as we cannot be sure which column is incorrect.

### Validity Check 5: Uniqueness Errors

**Type of error:** Duplicate records

**Description:** The combination of `ADDRESS` and `SALE DATE` should be unique because it is extremely unlikely that the same property is sold twice on the same day.

**How the error was introduced:** Randomly selected ~5% of rows and duplicated them, creating repeated `ADDRESS` + `SALE DATE` combinations.

In [15]:
# Introduce uniqueness errors by duplicating ~5% of rows
np.random.seed(5)
unique_err_df = df.copy()
dup_sample = unique_err_df.sample(frac=0.05, random_state=5)
unique_err_df = pd.concat([unique_err_df, dup_sample], ignore_index=True)

print(f"Rows before duplication: {len(df)}")
print(f"Rows after duplication: {len(unique_err_df)}")

Rows before duplication: 84548
Rows after duplication: 88775


In [16]:
def fix_address_sale_date_uniqueness(input_df):
    """
    Detect and return indices of duplicate `ADDRESS` + `SALE DATE` pairs.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'ADDRESS' and 'SALE DATE' columns.

    Returns:
        output_df (pd.DataFrame): The corrected DataFrame.
        dup_mask (pd.Series): A boolean mask of rows that were duplicated.
    """
    output_df = input_df.copy()
    dup_mask = output_df.duplicated(subset=["ADDRESS", "SALE DATE"], keep=False)

    # Correct by keeping the first occurrence of each duplicate pair
    output_df = output_df.drop_duplicates(subset=["ADDRESS", "SALE DATE"], keep="first")
    return output_df, dup_mask


# Run detection + correction
fixed_unique_df, dup_mask = fix_address_sale_date_uniqueness(unique_err_df)

print("Before correction:")
print(f"Duplicate rows detected: {dup_mask.sum()}")
print("Example duplicates:")
display(unique_err_df.loc[dup_mask, ["ADDRESS", "SALE DATE"]].head(3))
print(
    f"Column combo is unique: {(unique_err_df['ADDRESS'] + unique_err_df['SALE DATE']).is_unique}"
)
print("--------------------------------")
print("After correction:")
display(fixed_unique_df.loc[:, ["ADDRESS", "SALE DATE"]].head(3))
print(
    f"Column combo is unique: {(fixed_unique_df['ADDRESS'] + fixed_unique_df['SALE DATE']).is_unique}"
)

Before correction:
Duplicate rows detected: 16895
Example duplicates:


,ADDRESS,SALE DATE
22,244 EAST 7TH STREET,2017-06-21 00:00:00
23,244 EAST 7TH STREET,2017-06-21 00:00:00
63,"512 EAST 11TH STREET, 5C",2016-09-29 00:00:00


Column combo is unique: False
--------------------------------
After correction:


,ADDRESS,SALE DATE
0,153 AVENUE B,2017-07-19 00:00:00
1,234 EAST 4TH STREET,2016-12-14 00:00:00
2,197 EAST 3RD STREET,2016-12-09 00:00:00


Column combo is unique: True


**Qualitative results:** The detector flags duplicate `ADDRESS` + `SALE DATE` pairs. The example output shows repeated combinations before correction, and after dropping duplicates only one record per property per day remains.

### Validity Check 6: Presence Errors

**Type of error:** Missing values

**Description:** The `ZIP CODE` field should be present for each record. Missing values indicate incomplete data.

**How the error was introduced:** Randomly selected ~5% of rows and removed `ZIP CODE` values (set to missing).

In [17]:
# Introduce presence errors in ZIP CODE (~5% missing)
np.random.seed(6)
presence_df = df.copy()
presence_idx = presence_df.sample(frac=0.05, random_state=6).index

presence_df.loc[presence_idx, "ZIP CODE"] = np.nan

print("Example of missing ZIP CODE values:")
display(presence_df.loc[presence_idx, ["ZIP CODE"]].head(3))

Example of missing ZIP CODE values:


,ZIP CODE
22387,NaN
82999,NaN
8526,NaN


In [18]:
# Function to detect and correct presence errors in ZIP CODE
def fix_zipcode_presence(input_df):
    output_df = input_df.copy()
    missing_mask = output_df["ZIP CODE"].isna()

    # Correct by dropping rows with missing ZIP CODE
    output_df = output_df.loc[~missing_mask].copy()
    return output_df, missing_mask


# Run detection + correction
fixed_presence_df, missing_mask = fix_zipcode_presence(presence_df)

print(f"Rows with missing ZIP CODE detected: {missing_mask.sum()}")
print("Missing ZIP CODE examples:")
display(presence_df.loc[missing_mask, ["ZIP CODE"]].head(3))
print(f"After correction, removed {missing_mask.sum()} rows with missing ZIP CODE")

Rows with missing ZIP CODE detected: 4227
Missing ZIP CODE examples:


,ZIP CODE
7,NaN
18,NaN
38,NaN


After correction, removed 4227 rows with missing ZIP CODE


**Qualitative results:** The detector flags rows where `ZIP CODE` is missing. The example output shows missing entries before correction, and after dropping those rows the dataset contains only records with a valid ZIP code. Theoretically, this could be re-added based on the address, but that would require some external library to get address data.

### Validity Check 7: Length Errors

**Type of error:** Incorrect string length

**Description:** The `SALE DATE` field should have the same length as `2017-07-19 00:00:00` (19 characters). Any other length indicates a formatting issue.

**How the error was introduced:** Randomly selected ~5% of rows and modified `SALE DATE` strings to be too short or too long.

In [19]:
# Introduce length errors in SALE DATE (~5% with wrong length)
np.random.seed(7)
length_df = df.copy()
length_idx = length_df.sample(frac=0.05, random_state=7).index

# Make half too short, half too long
half = len(length_idx) // 2
short_idx = length_idx[:half]
long_idx = length_idx[half:]

length_df.loc[short_idx, "SALE DATE"] = (
    length_df.loc[short_idx, "SALE DATE"].astype(str).str.slice(0, 10)
)
length_df.loc[long_idx, "SALE DATE"] = (
    length_df.loc[long_idx, "SALE DATE"].astype(str) + "UTC-5"
)

print("Example of wrong-length SALE DATE values:")
display(length_df.loc[length_idx, ["SALE DATE"]].head(3))

Example of wrong-length SALE DATE values:


,SALE DATE
27481,2017-02-07
40363,2017-07-25
36122,2016-09-28


In [20]:
def find_erroneous_sale_date_length(input_df, expected_len=19):
    """
    Detect and return indices of incorrect string length in the 'SALE DATE' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'SALE DATE' column.
        expected_len (int): The expected length of the 'SALE DATE' string.

    Returns:
        wrong_len_mask (pd.Series): A boolean mask of rows that were incorrect.
    """
    output_df = input_df.copy()
    sale_str = output_df["SALE DATE"].astype(str)
    wrong_len_mask = sale_str.str.len() != expected_len

    return wrong_len_mask


# Run detection + correction
wrong_len_mask = find_erroneous_sale_date_length(length_df)
fixed_length_df = length_df[~wrong_len_mask]

print(f"Rows with wrong-length SALE DATE detected: {wrong_len_mask.sum()}")
print("Bad values before correction:")
display(length_df.loc[wrong_len_mask, ["SALE DATE"]].head(3))
print(
    f"After correction, removed {wrong_len_mask.sum()} rows with wrong-length SALE DATE"
)

Rows with wrong-length SALE DATE detected: 4227
Bad values before correction:


,SALE DATE
33,2016-12-02
36,2016-12-07
39,2017-06-27 00:00:00UTC-5


After correction, removed 4227 rows with wrong-length SALE DATE


**Qualitative results:** The detector flags `SALE DATE` strings that are not exactly 19 characters long (format 2017-07-19 00:00:00). The examples show truncated or extended values before correction, and after filtering only correctly formatted sale dates remain.

### Validity Check 8: Look-up Errors

**Type of error:** Invalid categorical value

**Description:** The `BUILDING CLASS CATEGORY` field should be one of the acceptable categories found in the dataset. Any value not in the allowed list is invalid.

**How the error was introduced:** Randomly selected ~5% of rows and replaced `BUILDING CLASS CATEGORY` with incorrect category names that are not in the acceptable list.

In [21]:
# Introduce look-up errors in BUILDING CLASS CATEGORY (~5%)
np.random.seed(8)
lookup_df = df.copy()
lookup_idx = lookup_df.sample(frac=0.05, random_state=8).index

# we pre-compute the list of acceptable values here based on the dataset,
# but in real life we would have to set this before aggregating the data
acceptable_values = list(df["BUILDING CLASS CATEGORY"].unique())

invalid_categories = [
    "99 UNKNOWN CATEGORY",
    "XX INVALID CLASS",
    "FAKE CATEGORY",
    "MISCELLANEOUS GROUP",
    "NOT A REAL CLASS",
]
lookup_df.loc[lookup_idx, "BUILDING CLASS CATEGORY"] = np.random.choice(
    invalid_categories, size=len(lookup_idx)
)

print("Example of invalid categories:")
display(lookup_df.loc[lookup_idx, ["BUILDING CLASS CATEGORY"]].head(3))

Example of invalid categories:


,BUILDING CLASS CATEGORY
61080,MISCELLANEOUS GROUP
58915,NOT A REAL CLASS
14291,XX INVALID CLASS


In [22]:
def find_erroneous_building_class_lookup(input_df, acceptable_values):
    output_df = input_df.copy()
    invalid_mask = ~output_df["BUILDING CLASS CATEGORY"].isin(acceptable_values)

    return invalid_mask


# Run detection + correction
acceptable_values = df["BUILDING CLASS CATEGORY"].unique()
invalid_mask = find_erroneous_building_class_lookup(lookup_df, acceptable_values)

print(f"Rows with invalid categories detected: {invalid_mask.sum()}")
print("Bad values before correction:")
display(lookup_df.loc[invalid_mask, ["BUILDING CLASS CATEGORY"]].head(3))
print(f"After correction, removed {invalid_mask.sum()} rows with invalid categories")

Rows with invalid categories detected: 4227
Bad values before correction:


,BUILDING CLASS CATEGORY
10,XX INVALID CLASS
34,XX INVALID CLASS
44,MISCELLANEOUS GROUP


After correction, removed 4227 rows with invalid categories


**Qualitative results:** The detector flags any `BUILDING CLASS CATEGORY` not in the acceptable list. The examples show invalid category names before correction, and after filtering only valid categories remain. In a real application, this would be set with a look up/dropdown instead of allowing people to type.